# Vertebal Classification

A vertebra (plural: vertebrae) is one of the series of small bones that form the vertebral column, also known as the spine or backbone.

Each vertebra is a bony unit that:

- Protects the spinal cord

- Supports the body’s weight

- Allows movement, bending, and rotation

The lumbar region refers to one of the lower parts of the spine and the lower back area of the body. As shown in the figure below, the lumbar region comprises five regions named L1, L2, L3, L4, and L5.

<div style="text-align: center;">
  <img src="lumbar-spine.png" alt="description">
</div>

In this task, we aim to develop a classification framework that categorises the 3D meshes of all five lumbar regions (L1, L2, L3, L4, and L5) into their corresponding anatomical classes. To do this, we need to implement the following sections: 
1. Importing the Required Packages
2. Loading Dataset
3. Normalising 3D shapes
4. Storing Points and Labels Into Arrays
5. Normalising 3D shapes
6. Loading Vector Extraction Using PCA
7. Developing the Classification Framework
8. Training and Testing the Classification Framework Using the Best Classifier and Number of Features

The dataset is saved in the data folder. All the meshes are saved in the VTK file. If you want to open these files, you can download and install paraview software from the link below: 

https://www.paraview.org/download/

## Installing Packages

Please run the below cell to install the required packages for this task. Once you run the cell, please restart the kernel and run the next cells.

In [3]:
%pip install -r requirements.txt

  Using cached trame_common-1.2.9-py3-none-any.whl.metadata (4.2 kB)
  Using cached trame_server-4.1.0-py3-none-any.whl.metadata (5.3 kB)
Using cached trame_common-1.2.9-py3-none-any.whl (24 kB)
Using cached trame_server-4.1.0-py3-none-any.whl (46 kB)

  Attempting uninstall: trame-common

    Found existing installation: trame-common 1.3.0

    Uninstalling trame-common-1.3.0:

      Successfully uninstalled trame-common-1.3.0

  Attempting uninstall: trame-server

    Found existing installation: trame-server 4.1.2

    Uninstalling trame-server-4.1.2:

      Successfully uninstalled trame-server-4.1.2

   ---------------------------------------- 2/2 [trame-server]

Note: you may need to restart the kernel to use updated packages.


## Importing the Required Packages 

In [4]:
import os 
import pyvista as pv # It is a package used for mesh and point cloud data 
import random
import numpy as np 
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Classifiers
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

from IPython.display import Image, display

pv.set_jupyter_backend("trame")   # now this should work

## Declaring the Dataset-related Varibales

In this section, we declare some dataset-related variables such as the direction of the dataset, the number of subjects, and the classes' names. 

In [5]:
data_dir = "data" 
n_subjects = 30 
vertebrae_classes = ["L1", "L2", "L3", "L4", "L5"] 

## Random Data Visualization

By running this cell, you can vislulise some of the mesh data using pyvista package

In [6]:
random_subjects = random.sample(range(1, n_subjects+1), 2)
random_subject_ids = [f"{x:02d}" for x in random_subjects]
random_subject_ids

# Colors for better visualization

colors = {
    "L1": "red",
    "L2": "orange",
    "L3": "green",
    "L4": "purple",
    "L5": "blue"
}

# Create 5 rows (vertebrae), 2 columns (subjects)
plotter = pv.Plotter(shape=(len(vertebrae_classes), 2), border=True)

# Loop through each vertebra (rows)
for row, vertebra in enumerate(vertebrae_classes):

    # Loop through subjects (columns)
    for col, sid in enumerate(random_subject_ids):
        
        # Switch to specific subplot
        plotter.subplot(row, col)

        # Compose filename
        filename = f"{vertebra}_{sid}.vtk"
        file_path = os.path.join(data_dir, vertebra, filename)
        
        # Read mesh
        mesh = pv.read(file_path)

        # Plot mesh
        plotter.add_mesh(mesh,
                         color=colors[vertebra],
                         opacity=0.9,
                         smooth_shading=True)

        # Add title
        plotter.add_text(f"{vertebra} - Subject {sid}", font_size=10)

        # Set good view
        plotter.camera_position = "iso"

plotter.show()

## Loading Dataset

This section contains a function named load_mesh_points_faces that reads a 3D mesh file with PyVista and outputs the points and the faces in the form of Numpy arrays. After defining this funtion, there is a loop that iterates over all subjects and all vertebra types to load their corresponding points and faces. This loop stores the data in a nested dictionary that its keys represent the subject IDS and their corresponding values contain the points and faces of L1, L2, L3, L4, and L5 shapes.   

In [7]:
def load_mesh_points_faces(filename):
    """
    Load a mesh using PyVista and return its vertices and triangular faces
    as NumPy arrays.

    Returns:
        points (np.ndarray): shape (N, 3)
        faces (np.ndarray): shape (M, 3)
    """
    
    # Read mesh with pyvista
    mesh = pv.read(filename)
    
    # Extract points (vertices)
    points = mesh.points.copy()  # shape: (N, 3)

    # Extract and reshape faces
    #
    # PyVista stores faces like:
    # [3, i, j, k, 3, i, j, k, ...]
    # where the leading 3 means triangle
    #
    raw_faces = mesh.faces.reshape(-1, 4)   # (M × 4)
    faces = raw_faces[:, 1:]                # remove "3" → keep only vertex indices
    
    return points, faces

In [8]:
all_data = dict() 
# Loop through each vertebra (rows)
n_subjects = 30 
for sub_id in range(1, n_subjects+1):
    sub_dict = dict()
    for row, vertebra in enumerate(vertebrae_classes):
        file_path = os.path.join(data_dir, vertebra, f"{vertebra}_{sub_id:02d}.vtk")
        points_arr, face_arr = load_mesh_points_faces(file_path)
        dict_data = {vertebra: {'points': points_arr, 'face': face_arr}}
        sub_dict.update(dict_data)
    all_data[sub_id] = sub_dict

In [9]:
print("Subject IDs stored in all_data:", all_data.keys())
random_sub_id = random.choice(list(all_data.keys()))
print(f"Vertebrae available for subject {random_sub_id}:", all_data[random_sub_id].keys())
print(f"Data fields stored for subject {random_sub_id}, vertebra L1:", all_data[random_sub_id]['L1'].keys())

Subject IDs stored in all_data: dict_keys([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30])
Vertebrae available for subject 19: dict_keys(['L1', 'L2', 'L3', 'L4', 'L5'])
Data fields stored for subject 19, vertebra L1: dict_keys(['points', 'face'])


## Normalising 3D shapes

In this section, there is a function named center_and_l2_normalize that that normalises each 3D shape.

Suppose $\mathbf{X} \in \mathbb{R}^{N \times 3}$ is a 3D point cloud where $N$ denotes the number of points. Given $\mathbf{X}$, the function should perform the following procedure.  

1. Compute the mean point of the 3D point cloud ($\mathbf{\tilde{X}}$)
2. Translate the 3D point cloud to the origin by $\mathbf{X} - \mathbf{\tilde{X}}$
3. Find the maximum L2 norm of the points of the translated 3D shape.
4. Divide the points of the translated point cloud by the maximum L2 norm. 

This function translates the 3D point cloud to the origin and normalises the shape to a cube of size 1.

After declaring the function, all the points of the dataset will be normalised through this function
In the next step, normalise all the points of the dataset using this function.


In [10]:
## Normalising Function

def center_and_l2_normalize(points):
    """
    Translate point cloud centroid to (0,0,0)
    and normalize each vertex by its Euclidean norm.
    
    Args:
        points (numpy.ndarray): shape (N, 3) point cloud
    
    Returns:
        numpy.ndarray: transformed point cloud
    """
    # Step 1: move centroid to origin
    centroid = np.mean(points, axis=0)
    centered = points - centroid

    # Step 2: find max distance from origin
    distances = np.linalg.norm(centered, axis=1)
    max_dist = np.max(distances)

    # Step 3: normalize so farthest point lies on unit sphere
    normalized = centered / (max_dist + 1e-8)  # avoid divide by zero
    
    return normalized

In [11]:
## Normalising all the points of the dataset using center_and_l2_normalize function

all_data_norm = all_data.copy()

for sub_id in all_data_norm.keys():
    for vert_class in vertebrae_classes:
        sub_points = all_data_norm[sub_id][vert_class]['points']
        sub_points_norm = center_and_l2_normalize(sub_points)
        all_data_norm[sub_id][vert_class]['points'] = sub_points_norm

## Storing Points and Labels Into Arrays

In this section, you will store the points and the labels of the dataset into $\mathbf{X_{array}} \in \mathbb{R}^{150 \times 12000}$ and $\mathbf{Y_{array}} \in \mathbb{R}^{150}$, respectively (30 subjects × 5 vertebrae = 150 shapes).

To do this, you have the initial lists for X and Y, and you flatten the point arrays. Then, you append the flattened point arrays to X. 

You will also apply the vertebra class names to the mapping_vert_classes function to get the coded labels for each data point. According to this function, the cases in the L1, L2, L3, L4, and L5 classes have labels 0, 1, 2, 3, and 4, respectively. 

After looping through and gathering the points and labels into X and Y, these two lists are transformed into NumPy arrays using the np.asarray() function to get  $\mathbf{X_{array}}$ and $\mathbf{Y_{array}}$. 

In [12]:
X = [] 
Y = [] 

def mapping_vert_classes(vert_class): 
    
    if vert_class == 'L1':
        num_class = 0 
    elif vert_class == 'L2': 
        num_class = 1
    elif vert_class == 'L3': 
        num_class = 2
    elif vert_class == 'L4': 
        num_class = 3
    elif vert_class == 'L5':
        num_class = 4 
        
    return num_class 

for sub_id in all_data_norm.keys():
    for vert_class in vertebrae_classes: 
        mesh_points = all_data_norm[sub_id][vert_class]['points']
        mesh_faces = all_data_norm[sub_id][vert_class]['face']

        X.append(mesh_points.flatten())
        Y.append(mapping_vert_classes(vert_class))

X_array = np.asarray(X)
Y_array = np.asarray(Y)

In [13]:
# Added explanations
print("Shape of X_array (number of samples × flattened vertex coordinates):", X_array.shape)
print("Shape of Y_array (number of labels):", Y_array.shape)

Shape of X_array (number of samples × flattened vertex coordinates): (150, 12000)
Shape of Y_array (number of labels): (150,)


## Loading Vector Extraction Using PCA

**Now you need to write a function named loading_vector_extarction that performs PCA transformation on the feature matrix, $\mathbf{X} \in \mathbb{R}^{N \times M}$, where $N$ is the number of data and $M$ is the number of features. For our case, $N=150$ and $M=12000$, and returns the principal components of the dataset.** 

The principal components, also called loading vectors, indicate the directions of maximum variance in a dataset, ordered from highest to lowest. 

To write the function, please conduct the following steps. 
1. Compute the mean of the $\mathbf{X}$ regarding the number of data.
2. Subtract the $\mathbf{X}$ from the obtained mean.
3. Fit PCA on the subtracted data given the number of components.
4. Return the PCA components, mean vector, and singular values, respectively.

To define this function, you use the PCA function from the scikit-learn package, which is explained in: https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html

After defining this function, run it on X_array with n_components=90 using the two next cells. 

In [14]:
def loading_vector_extarction(X_array, n_components):
    """
    Perform PCA on the input matrix and extract loading vectors,
    the global mean shape, and singular values.

    Inputs:
        X_array (np.ndarray):
            A data matrix of shape (N, M), where N is the number of samples
            and M is the number of features (e.g., flattened mesh points).
        
        n_components (int):
            The number of principal components (loading vectors) to compute.

    Outputs:
        Vt (np.ndarray):
            The PCA loading vectors (principal component directions),
            with shape (n_components, M).

        X_mean (np.ndarray):
            The mean vector of the dataset, shape (M,). This is subtracted
            from the data before PCA.

        S (np.ndarray):
            The singular values of the PCA decomposition, one per component.
    """

    ### Write your code here 


    return Vt, X_mean, S

In [15]:
Vt, X_mean, S = loading_vector_extarction(X_array, n_components=90)

NameError: name 'Vt' is not defined

In [ ]:
print("Shape of Vt (loading vectors = number of components × number of features):", Vt.shape)
print("Shape of X_mean (mean vector = one value per feature):", X_mean.shape)
print("Shape of X_array (original data = number of samples × number of features):", X_array.shape)
print("Shape of (X_array - X_mean) (mean-centered data):", (X_array - X_mean).shape)

## Shape Generation Using the First and Second Loading Vectors (Principal Components)

This section aims to visulise the geenrated shapes using the first and second Loading Vectors (Principal Components). The directions of changes that the generated shapes show are the same as the first and second Loading Vectors (Principal Components). 

In [ ]:
# Shape Generation Using the First Loading Vector
largest_eigen_value = S[0]
r_range = np.arange(-3 * np.sqrt(largest_eigen_value),
                    3 * np.sqrt(largest_eigen_value),
                    np.sqrt(largest_eigen_value) / 10)

gen_shapes = [] 
for r in r_range: 
    gen_shape = X_mean + (r * Vt[0, :])
    gen_shapes.append(gen_shape.reshape(4000, 3))

faces = all_data_norm[1]['L1']['face']
n_faces = faces.shape[0]
faces_pv = np.hstack(
    [np.full((n_faces, 1), 3, dtype=np.int64), faces.astype(np.int64)]
).ravel()

# Create initial mesh
mesh = pv.PolyData(gen_shapes[0], faces_pv)

# Use off_screen=True for Jupyter
plotter = pv.Plotter(off_screen=True)
plotter.add_mesh(mesh, color="lightgray", show_edges=True)

gif_name = "mesh_sequence_PC1.gif"
plotter.open_gif(gif_name)

for pts in gen_shapes:
    mesh.points = pts
    plotter.render()
    plotter.write_frame()

plotter.close()

# Display the GIF in the notebook
display(Image(filename=gif_name))

In [ ]:
second_largest_eigen_value = S[1]
r_range_PC2 = np.arange(-3*np.sqrt(second_largest_eigen_value), 3*np.sqrt(second_largest_eigen_value))
gen_shapes = [] 
for r in r_range_PC2: 
    gen_shape = X_mean + (r * Vt[1,:])
    gen_shapes.append(gen_shape.reshape(4000,3))

faces = all_data_norm[1]['L1']['face']
n_faces = faces.shape[0]
faces_pv = np.hstack(
    [np.full((n_faces, 1), 3, dtype=np.int64), faces.astype(np.int64)]
).ravel()

mesh = pv.PolyData(gen_shapes[0], faces_pv)

plotter = pv.Plotter()
plotter.add_mesh(mesh, color="lightgray", show_edges=True)

gif_name = "mesh_sequence_PC2.gif"
plotter.open_gif(gif_name)

for pts in gen_shapes:
    mesh.points = pts
    plotter.render()
    plotter.write_frame()

plotter.close()

display(Image(filename=gif_name))

## Developing the Classification Framework

In this section, we aim to develop a classification framework that assigns each vertebral mesh in our dataset to one of the L1, L2, L3, L4, or L5 classes.

Each subject contributes five vertebrae to the dataset. Therefore, the training and testing subsets must be created using **subject-wise data splitting**, so that all vertebrae from a given subject are assigned entirely to either the training set or the testing set. This prevents anatomical information from the same subject from appearing in both subsets and provides a more appropriate evaluation of generalisation to unseen subjects.

Use `GroupShuffleSplit` to assign approximately 70% of the subjects to the training subset and 30% to the testing subset.

Since the original mesh representation has a very large number of features (12000), PCA-based feature reduction will be used before classification. We consider a range of 10 to 55 PCA components and three classifiers: KNN, linear SVM, and logistic regression. The `feature_matrix_cal` function projects the data onto the obtained principal components.

**Your main objective is to find the optimal number of PCA components and the optimal classifier that achieve the highest classification accuracy on the subject-wise testing set.**

After finding the best configuration, you will also investigate the **effect of PCA-based feature reduction** by training the same best classifier directly on the original high-dimensional features and comparing its performance with the PCA-based representation.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# -------------------------------------------------------
# Create subject IDs
# -------------------------------------------------------
# X_array and Y_array were constructed subject-by-subject.
# Each subject contributes five vertebrae (L1--L5).
subject_ids = np.repeat(
    np.asarray(list(all_data_norm.keys())),
    len(vertebrae_classes)
)

assert len(subject_ids) == len(X_array) == len(Y_array)

# -------------------------------------------------------
# Subject-wise train-test split
# -------------------------------------------------------
group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.3,
    random_state=42
)

### Write your code here
# Obtain train_idx and test_idx using:
# X_array, Y_array and groups=subject_ids


### Write your code here
# Use train_idx and test_idx to create:
# X_train, X_test, Y_train, Y_test
# subject_ids_train and subject_ids_test


### Write your code here
# Verify that no subject appears in both training and testing sets.


print("X_train shape (training samples × features):", )
print("X_test shape  (test samples × features):", )
print("Y_train shape (training labels):", )
print("Y_test shape  (test labels):", )

n_components = np.arange(10, 60, 5)
print("List of PCA components to evaluate:", n_components)

# -------------------------------------------------------
# Define classifiers to compare
# -------------------------------------------------------
classifiers = {
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "LinearSVM": SVC(kernel="linear", C=1, decision_function_shape="ovr"),
    "LR": LogisticRegression()
}


In [ ]:
def feature_matrix_cal(X_array, Vt, X_mean):
    """
    Compute the PCA feature matrix by projecting samples onto
    the PCA loading vectors.

    Inputs:
        X_array (np.ndarray):
            The original data matrix of shape (N, M),
            where N is the number of samples and
            M is the number of features (e.g., flattened mesh coordinates).

        Vt (np.ndarray):
            The PCA loading vectors (principal components),
            with shape (K, M), where K is the number of components used.

        X_mean (np.ndarray):
            The mean vector of the training data, shape (M,).
            This is used to mean-center the samples before projection.

    Output:
        X_feature_matrix (np.ndarray):
            The PCA feature matrix of shape (N, K).
            Each row contains the K PCA coefficients (scores) for one sample.
            This represents the lower-dimensional encoding of the data.
    """

    X_feature_matrix = Vt @ (X_array - X_mean).T
    X_feature_matrix = X_feature_matrix.T
    return X_feature_matrix

**You need to complete the cell below to find the best and worst number of components. You can use the provided template to complete the cell.** 

In [ ]:
results = {}

# -------------------------------------------------------
# Evaluate all numbers of components with all classifiers
# -------------------------------------------------------
for each_num_components in n_components:

    ### Write your code here 

    """
    Apply PCA Feature reduction here 
    """

    ## Use standard scaling for normalisation on both training and testing sets
    scaler = StandardScaler()
    X_train_transformed = scaler.fit_transform()
    X_test_transformed  = scaler.transform()

    ## You can use the results variable to save the accaracies for each number of components and classifier 
    results[int(each_num_components)] = {}

    ## Declaring the classfication models 
    knn_model = KNeighborsClassifier(n_neighbors=5)
    svm_model = SVC(kernel="linear", C=1, decision_function_shape="ovr")
    lr_model  = LogisticRegression()

    """
    Train and test the classifiers 
    """

    results[int(each_num_components)]["KNN"]       = knn_acc
    results[int(each_num_components)]["LinearSVM"] = svm_acc
    results[int(each_num_components)]["LR"]        = lr_acc

    print(f"Components={each_num_components}, Classifier = KNN,       Accuracy={knn_acc:.4f}")
    print(f"Components={each_num_components}, Classifier = LinearSVM, Accuracy={svm_acc:.4f}")
    print(f"Components={each_num_components}, Classifier = LR,        Accuracy={lr_acc:.4f}")

# -------------------------------------------------------
# Find the global best AND worst classifier and component count
# -------------------------------------------------------
best_components   = None
best_classifier   = None
best_accuracy     = -1.0

worst_components  = None
worst_classifier  = None
worst_accuracy    = 1.0  # start from 1 (max possible accuracy)

for comp, clf_dict in results.items():
    for clf_name, acc in clf_dict.items():
        # Check for best
        if acc > best_accuracy:
            best_accuracy   = acc
            best_classifier = clf_name
            best_components = comp
        # Check for worst
        if acc < worst_accuracy:
            worst_accuracy   = acc
            worst_classifier = clf_name
            worst_components = comp

print("\n========================")
print("Best configuration found")
print("========================")
print(f"Best number of components : {best_components}")
print(f"Best classifier           : {best_classifier}")
print(f"Best accuracy             : {best_accuracy:.4f}")

print("\n========================")
print("Worst configuration found")
print("========================")
print(f"Worst number of components: {worst_components}")
print(f"Worst classifier          : {worst_classifier}")
print(f"Worst accuracy            : {worst_accuracy:.4f}")


**In the cell below, you need to write a code to plot the obtained accuracies vs the number of components for the classifiers. You can use the template to complete the cell.**

In [ ]:
# Sort components so the x-axis is ordered
components = sorted(results.keys())

# Extract accuracies for each classifier in the same order
knn_acc = [results[c]["KNN"] for c in components]
svm_acc = [results[c]["LinearSVM"] for c in components]
LR_acc = [results[c]["LR"] for c in components]

"""
Write the plotting code 
"""

**Please train and test the classification model using the best classifier and the best number of components, and plot the confusion matrix give the testing set**

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


### Write your code 


**Please train and test the classification model using the worst classifier and the worst number of components and plot the confusion matrix give the testing set**

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


### Write your code 

## Effect of PCA-Based Feature Reduction on Classification Performance

The classification framework above uses PCA to reduce the dimensionality of the original vertebral mesh representation before training the classifier. In this section, we investigate the effect of this dimensionality reduction on classification performance.

To ensure a fair comparison, use:

- the **same subject-wise training and testing subsets** created earlier;
- the **same best classifier** identified in the PCA-based experiment;
- the same class labels.

The only difference between the two experiments should be the feature representation:

Complete the cell below to:

1. Standardise the original high-dimensional `X_train` and `X_test` feature matrices.
2. Train the previously selected `best_classifier` directly on the original features.
3. Calculate the classification accuracy without PCA.
4. Compare this accuracy with the accuracy obtained using the optimal PCA-based representation.
5. Report the original and PCA-reduced feature dimensionalities.
6. Generate the confusion matrix for the classifier trained on the original high-dimensional features.
7. Generate the confusion matrix for the classifier trained on the optimal PCA-reduced feature representation.
8. Compare the two confusion matrices and identify which vertebral classes benefit most, or least, from PCA-based feature reduction.

**Question:** Compare the classification performance obtained using the original high-dimensional feature representation with that obtained using PCA-reduced features. Consider both the overall accuracy and the class-specific errors shown in the confusion matrices. Does PCA improve classification performance? Briefly discuss the result in terms of dimensionality, redundant information, noise, overfitting, and class separability.


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

# -------------------------------------------------------
# Classification using the original feature representation
# -------------------------------------------------------

### Write your code here
# Standardise X_train and X_test using StandardScaler.
# Remember to fit the scaler only on the training data.


# -------------------------------------------------------
# Initialise the previously selected best classifier
# -------------------------------------------------------
clf_models = {
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "LinearSVM": SVC(kernel="linear", C=1, decision_function_shape="ovr"),
    "LR": LogisticRegression()
}

### Write your code here
# Select the classifier corresponding to best_classifier.


# -------------------------------------------------------
# Train and evaluate using the original features
# -------------------------------------------------------
### Write your code here
# Fit the classifier using the standardised original
# training features, predict the test labels, and calculate
# raw_accuracy.


# -------------------------------------------------------
# Re-train the best classifier using the optimal PCA features
# -------------------------------------------------------
### Write your code here
# Use best_components to obtain PCA loading vectors from X_train.
# Project X_train and X_test using feature_matrix_cal.
# Standardise the PCA feature matrices.
# Train the same best classifier and obtain Y_pred_pca
# and pca_accuracy.


# -------------------------------------------------------
# Compare original and PCA-reduced representations
# -------------------------------------------------------
print("Original feature dimensionality :", )
print("PCA feature dimensionality      :", best_components)

print("\n========================================")
print("Effect of PCA-based feature reduction")
print("========================================")
print(f"Classifier                 : {best_classifier}")
print(f"Without PCA accuracy       : {raw_accuracy:.4f}")
print(f"With PCA accuracy          : {pca_accuracy:.4f}")
print(f"Accuracy difference        : {pca_accuracy - raw_accuracy:+.4f}")


# -------------------------------------------------------
# Confusion matrix: original features
# -------------------------------------------------------
### Write your code here
# Compute and display the confusion matrix obtained from Y_pred_raw.


# -------------------------------------------------------
# Confusion matrix: PCA-reduced features
# -------------------------------------------------------
### Write your code here
# Compute and display the confusion matrix obtained from Y_pred_pca.
